# Moving Averages for Decomposition

In Chapter 02 we used moving averages for **smoothing** — revealing trends by
filtering out noise.  Here the focus shifts to their role in **decomposition**:
extracting the trend-cycle component $\hat{T}_t$ from a seasonal time series.

Classical decomposition starts by estimating the trend with a moving average,
then isolates the seasonal component from the de-trended residual.  The
particular type of MA used — the **$2 \times m$ moving average** — is designed
to handle even-order seasonality (monthly data with $m=12$, quarterly with
$m=4$), and this notebook develops it step by step.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]
passengers = airline["Passengers"]

print("Shape:", airline.shape)
airline.head()

---
## 1. Simple Moving Average as a Trend Estimator

A **simple moving average** (SMA) of order $m$ computes the mean of $m$
consecutive observations.  When $m$ matches the seasonal period, the MA
averages over a full cycle and the seasonal pattern cancels out, leaving an
estimate of the **trend-cycle** component.

For monthly data with annual seasonality, $m = 12$. For quarterly data,
$m = 4$.

$$
\hat{T}_t = \frac{1}{m} \sum_{j=0}^{m-1} y_{t-j}
$$

This is different from using an MA for smoothing (where the window might be
chosen to reduce noise) — here the window is specifically chosen to **eliminate
seasonal variation**.

In [ ]:
# 12-month trailing SMA — removes annual seasonality
sma_12 = passengers.rolling(window=12).mean()

fig, ax = plt.subplots()
ax.plot(passengers, label="Original", alpha=0.6)
ax.plot(sma_12, label="12-month SMA (trailing)", linewidth=2.5)
ax.set_title("12-Month Moving Average as Trend Estimator")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The 12-month MA produces a clean trend line with no seasonal fluctuation.")
print(f"Note: the first {11} values are NaN (insufficient data for the window).")

---
## 2. Choosing the Order of the MA

The **order** $m$ should match the seasonal period:

| Data frequency | Seasonal period | MA order $m$ |
|---|---|---|
| Monthly | Annual (12 months) | 12 |
| Quarterly | Annual (4 quarters) | 4 |
| Weekly | Annual (52 weeks) | 52 |
| Daily (business) | Weekly (5 days) | 5 |

Using a window that does not match the seasonal period will **not** fully
remove seasonality, leaving residual oscillations in the trend estimate.

In [ ]:
# Compare different window sizes on monthly data
windows = [4, 6, 12, 24]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, w in zip(axes.flat, windows):
    sma = passengers.rolling(window=w).mean()
    ax.plot(passengers, alpha=0.4, label="Original")
    ax.plot(sma, linewidth=2, label=f"{w}-month MA")
    ax.set_title(f"Window = {w}")
    ax.legend(loc="upper left")
    ax.set_ylabel("Thousands")

plt.suptitle("Effect of Window Size on Trend Estimation", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Window = 12 cleanly removes annual seasonality.")
print("Windows of 4 and 6 leave residual seasonal oscillations.")
print("Window = 24 over-smooths, lagging behind the true trend.")

---
## 3. The $2 \times m$ Moving Average

### The symmetry problem with even-order MAs

When $m$ is **odd** (e.g., 5), a centred MA is symmetric: two observations on
each side plus the centre point.  Each value of $\hat{T}_t$ aligns with a
single time point.

When $m$ is **even** (e.g., 12 for monthly data), a centred MA falls
**between** two time points.  A 12-MA centred at $t$ needs observations from
$t-6$ to $t+5$ (or $t-5$ to $t+6$) — either way it is asymmetric.

### The solution: apply a 2-MA to the $m$-MA

To restore symmetry, we apply a **second moving average of order 2** to the
result of the first $m$-MA.  This "$2 \times m$-MA" averages two adjacent
$m$-MA values, producing a result that is centred exactly on a time point.

The combined filter is:

$$
\hat{T}_t = \frac{1}{2m}\left(y_{t-m/2} + 2\sum_{j=-(m/2-1)}^{m/2-1} y_{t+j} + y_{t+m/2}\right)
$$

The first and last observations in the window receive **half weight**, while
all interior observations receive **full weight**.  This makes the filter
symmetric and centred.

In [ ]:
# Step 1: 12-MA (centred)
ma_12 = passengers.rolling(window=12, center=True).mean()

# Step 2: 2-MA on top of the 12-MA -> 2x12-MA
ma_2x12 = ma_12.rolling(window=2, center=True).mean()

# Pandas shortcut: trailing 12-MA then trailing 2-MA
# (equivalent result, just shifted)
ma_2x12_trailing = passengers.rolling(12).mean().rolling(2).mean()

print("Step-by-step 2x12-MA (first 15 values):")
comparison = pd.DataFrame({
    "Original": passengers,
    "12-MA (centred)": ma_12,
    "2x12-MA (centred)": ma_2x12,
})
comparison.head(15)

In [ ]:
# Visualise the step-by-step construction
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(passengers, alpha=0.4, label="Original")
ax.plot(ma_12, label="12-MA (centred)", linestyle=":", linewidth=2)
ax.plot(ma_2x12, label="$2 \\times 12$-MA (centred)", linewidth=2.5)
ax.set_title("Constructing the $2 \\times 12$ Moving Average")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The 12-MA and 2x12-MA are very similar, but the 2x12-MA is properly")
print("centred on integer time points rather than falling between them.")

### Manual verification of the $2 \times 12$-MA formula

Let us verify the formula by computing one value by hand and comparing it to
the pandas result.

In [ ]:
# Manual computation for t = index position 6 (July 1949)
# The 2x12-MA at t uses observations from t-6 to t+6
# with half weight on endpoints, full weight on interior
m = 12
t = 6  # index position
vals = passengers.values

# Apply the formula: (y[t-6] + 2*sum(y[t-5:t+6]) + y[t+6]) / (2*12)
half_m = m // 2
manual = (vals[t - half_m]
          + 2 * np.sum(vals[t - half_m + 1 : t + half_m])
          + vals[t + half_m]) / (2 * m)

pandas_val = ma_2x12.iloc[t]

print(f"Date: {passengers.index[t].strftime('%Y-%m')}")
print(f"Manual formula result:  {manual:.4f}")
print(f"Pandas 2x12-MA result: {pandas_val:.4f}")
print(f"Match: {np.isclose(manual, pandas_val)}")

---
## 4. Weighted Moving Averages

A **weighted moving average** assigns different importance to different
positions in the window.  The $2 \times m$-MA is actually a special case:
the endpoints get weight $\frac{1}{2m}$ while interior points get weight
$\frac{1}{m}$.

In general, a weighted MA of order $2k+1$ is:

$$
\hat{T}_t = \sum_{j=-k}^{k} a_j \, y_{t+j}
$$

where $\sum a_j = 1$.  The weights must be **symmetric** ($a_j = a_{-j}$) for
the filter to be centred.

In [ ]:
# The implicit weights of a 2x12-MA
# Endpoints: 1/(2*12) = 1/24
# Interior:  2/(2*12) = 1/12
weights_2x12 = np.array([1/24] + [1/12]*11 + [1/24])
print(f"Number of weights: {len(weights_2x12)}")
print(f"Sum of weights: {weights_2x12.sum():.6f}")
print(f"Weights: {np.round(weights_2x12, 4)}")

In [ ]:
# Visualise the weight profile of the 2x12-MA
fig, ax = plt.subplots(figsize=(10, 4))

positions = np.arange(-6, 7)
ax.bar(positions, weights_2x12, color="tab:blue", edgecolor="k", alpha=0.7)
ax.set_title("Weights of the $2 \\times 12$ Moving Average")
ax.set_xlabel("Position relative to centre ($j$)")
ax.set_ylabel("Weight ($a_j$)")
ax.set_xticks(positions)
plt.tight_layout()
plt.show()

print("Endpoints (j = -6 and j = +6) receive half the weight of interior points.")
print("This is what makes the filter symmetric and centred.")

In [ ]:
# Compare uniform (simple) MA vs 2x12 weighted MA
# A uniform 13-point centred MA for comparison
uniform_13 = passengers.rolling(window=13, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(passengers, alpha=0.3, label="Original")
ax.plot(ma_2x12, label="$2 \\times 12$-MA (weighted)", linewidth=2)
ax.plot(uniform_13, label="Uniform 13-MA (centred)", linewidth=2, linestyle="--")
ax.set_title("$2 \\times 12$-MA vs Uniform 13-MA")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("Both are 13-point symmetric filters, but the 2x12-MA is specifically")
print("designed to remove seasonality of period 12.")

---
## 5. Moving Averages as Trend Estimators

The primary purpose of the MA in decomposition is to produce $\hat{T}_t$ — the
estimated trend-cycle.  Key properties:

- The MA **removes seasonality** when the window matches the seasonal period.
- The MA **smooths noise**, producing a cleaner trend estimate.
- **Edge effects**: the first and last $m/2$ observations are `NaN` because
  there is not enough data to fill the window.  This is an inherent limitation
  of centred MAs.

In [ ]:
# 12-month MA as trend estimator: highlight edge effects
trend = ma_2x12.copy()

nan_count_start = trend.isna().sum() - trend[::-1].isna().sum()  # approximate
total_nans = trend.isna().sum()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(passengers, alpha=0.4, label="Original")
ax.plot(trend, label="$2 \\times 12$-MA trend", linewidth=2.5, color="tab:red")

# Shade the NaN regions
first_valid = trend.first_valid_index()
last_valid = trend.last_valid_index()
ax.axvspan(passengers.index[0], first_valid, alpha=0.15, color="grey", label="Edge effect (NaN)")
ax.axvspan(last_valid, passengers.index[-1], alpha=0.15, color="grey")

ax.set_title("Trend Estimate with Edge Effects Highlighted")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total observations: {len(passengers)}")
print(f"NaN values in trend: {total_nans}")
print(f"Usable trend estimates: {len(passengers) - total_nans}")

In [ ]:
# Compare different window sizes as trend estimators
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(passengers, alpha=0.3, label="Original")

for w, color in [(6, "tab:green"), (12, "tab:orange"), (24, "tab:red")]:
    ma = passengers.rolling(window=w, center=True).mean()
    ax.plot(ma, label=f"{w}-month MA", linewidth=2, color=color)

ax.set_title("Trend Estimates with Different Window Sizes")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("6-month MA: retains some seasonal oscillation (window < seasonal period).")
print("12-month MA: removes annual seasonality cleanly.")
print("24-month MA: over-smooths — the trend lags behind turning points.")

---
## 6. Moving Average of Moving Average

The **$2 \times 12$-MA** used in classical decomposition is computed in two
steps:

1. Apply a **12-MA** (trailing): `passengers.rolling(12).mean()`
2. Apply a **2-MA** (trailing) to the result: `.rolling(2).mean()`

This is exactly what `statsmodels.tsa.seasonal.seasonal_decompose` does
internally.  The chained call produces a symmetric, centred filter even though
each individual rolling call is trailing.

In [ ]:
# The 2x12-MA via chained trailing rolling calls
step1 = passengers.rolling(12).mean()         # 12-MA trailing
step2 = step1.rolling(2).mean()               # 2-MA trailing

# This is equivalent to the centred 2x12-MA (shifted by the appropriate offset)
print("Step 1 — 12-MA (first 15 values):")
print(step1.head(15).to_string())
print()
print("Step 2 — 2x12-MA (first 15 values):")
print(step2.head(15).to_string())

In [ ]:
# Visual comparison: chained trailing vs centred approach
centred_2x12 = passengers.rolling(12, center=True).mean().rolling(2, center=True).mean()
trailing_2x12 = passengers.rolling(12).mean().rolling(2).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(passengers, alpha=0.3, label="Original")
ax.plot(centred_2x12, label="$2 \\times 12$-MA (centred)", linewidth=2.5)
ax.plot(trailing_2x12, label="$2 \\times 12$-MA (trailing)", linewidth=2, linestyle="--")
ax.set_title("Centred vs Trailing $2 \\times 12$-MA")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The trailing version is shifted to the right by 6 months.")
print("For decomposition we need the centred version (or we shift the result back).")

In [ ]:
# Verify that statsmodels uses this exact approach internally
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(passengers, model="additive", period=12)
sm_trend = decomp.trend

# Compare to our centred 2x12-MA
diff = (centred_2x12 - sm_trend).dropna()
print(f"Max difference between our 2x12-MA and statsmodels trend: {diff.abs().max():.2e}")
print("They are identical — classical decomposition uses the 2x12-MA as its trend estimator.")

In [ ]:
# Final overview: the decomposition pipeline
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

axes[0].plot(passengers)
axes[0].set_title("Original Series")
axes[0].set_ylabel("Passengers")

axes[1].plot(decomp.trend, color="tab:orange")
axes[1].set_title("Trend ($2 \\times 12$-MA)")
axes[1].set_ylabel("Trend")

axes[2].plot(decomp.seasonal, color="tab:green")
axes[2].set_title("Seasonal Component")
axes[2].set_ylabel("Seasonal")

axes[3].plot(decomp.resid, color="tab:red")
axes[3].set_title("Residual")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")

plt.suptitle("Classical Additive Decomposition — Built on the $2 \\times 12$-MA",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Everything starts with the 2x12-MA trend estimate.")
print("The seasonal component is the average of (original - trend) for each month.")
print("The residual is what remains after removing trend and season.")

---
## Summary

| Concept | Key point |
|---|---|
| **MA order** | Must match the seasonal period ($m=12$ for monthly, $m=4$ for quarterly) |
| **Even-order problem** | A single $m$-MA with even $m$ is not centred on a time point |
| **$2 \times m$-MA** | Applies a 2-MA on top of an $m$-MA to create a symmetric, centred filter |
| **Weight structure** | Endpoints get half weight ($\frac{1}{2m}$); interior gets full weight ($\frac{1}{m}$) |
| **Edge effects** | First and last $m/2$ observations are lost (NaN) |
| **Pandas implementation** | `df.rolling(12).mean().rolling(2).mean()` |
| **Connection to decomposition** | `seasonal_decompose` uses the $2 \times m$-MA as its trend estimator |